# mini-beatrix-2 — the governed full-splat mission

**Craft**: `mini-beatrix-2` — d1024 × L32, ctx 8192, a governed CausalSplatHUB in **every** block
(16 constellations × 256 anchors @ D=256 each, supply 1.0×D), 6 banks @ ff1024, ~849M params.
Plan of record: claude-mind `history/plans/2026-08-26_mini_beatrix_v2_shape.md` · sizing console: Beatrix Foundry.

**Laws embodied** (all measured): supply K ≤ 2·D per book (ROUND 5e) · anchor governor from birth
(min-sep projection, post-step, identity-when-slack — ROUND 5f) · product-code composition by budget,
never softmax over books (B4) · pure Adam + Muon split, wd 0 · bf16-train / fp8-ship / never-fp16
(squared address terms flush) · crash-safe resume-first · **per-boundary report + checkpoint pushes
(the ship-complete law — no training without push cells)**.

**Binding rider**: the additive hub write saturates at high demand (R=64: .84 vs .99). The delta-write
screen is queued separately; until it passes, no claim here covers retrieval.

Target hardware: RTX 6000 Pro Blackwell 96GB. `TURBO` stays False — compiled bf16 backward NaNs on
Blackwell (measured); eager-fused is the training path.

Recommended first session: run Cell P once with `PRESET='mini-beatrix-2s'` (the 237M screen craft) —
the gating cells (full-splat causal viability at 2ep-scale budgets) are cheapest there.

In [ ]:
# C1 — pinned installs. RESTART RUNTIME if the version line below changes.
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # Linux Colab: kills the fragmentation tax
%pip install -q "datasets>=5.0.0" "geolip-alephllm @ git+https://github.com/AbstractEyes/alephllm@2be28dd99c33036dba6f95b44d4e7d51237b9344"
import geolip.alephllm as A
print('geolip.alephllm', A.__version__)
assert tuple(int(x) for x in A.__version__.split('.')) >= (0, 7, 2), (
    'RESTART REQUIRED: runtime still holds the old package — Runtime > Restart, then rerun from C1')

In [ ]:
# C2 (P) — PREFLIGHT: build, law check, bit-exact births, bench gate. No training.
import time, torch
from google.colab import userdata
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.model.governor import govern_model

HF_TOKEN = userdata.get('HF_TOKEN')
PRESET = 'mini-beatrix-2'          # 'mini-beatrix-2s' for the screen craft; '-control' twins exist
p = get_preset(PRESET); cfg = p.model; tc = p.train
torch.manual_seed(tc.seed)
model = AlephLM(cfg).cuda()
n = model.param_count()
print(f'{cfg.name}: {n/1e6:.1f}M params | hubs {len(cfg.hub_layers)}/{cfg.n_layers} blocks × '
      f'{cfg.hub_const} books × {cfg.hub_K}@D{cfg.hub_D} (supply {cfg.hub_K/cfg.hub_D:.2f}×D) | ctx {cfg.context}')
assert cfg.hub_K <= 2 * cfg.hub_D, 'supply law violated — this preset should not exist'
hits = govern_model(model, tc.governor_theta)
print(f'governor birth check: {hits} hits (must be 0 at mission D)'); assert hits == 0
x = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
with torch.no_grad():   # C6 null paths: bank + head aleph contribute exactly zero at init
    a = model(x, disable_bank=True, disable_head_aleph=True).logits
    b = model(x).logits
    assert torch.equal(a, b), 'C6 violated — born-null paths are not null'
print('C6 null paths bit-exact at birth')
opt_probe = torch.optim.SGD(model.parameters(), lr=0.0)
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
for i in range(3):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        out = model(x, targets=x)
    out.loss.backward(); opt_probe.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sps = (time.time() - t0) / 3
vram = torch.cuda.max_memory_allocated() / 2**30
toks = tc.micro_batch * cfg.context
print(f'bench: {sps:.2f}s / micro-step · {toks/sps/1e3:.0f}k tok/s (×{tc.grad_accum} accum) · peak {vram:.1f} GB')
assert vram < 88, f'over the Blackwell budget at micro_batch={tc.micro_batch} — lower it in the preset'
del model, out, opt_probe; torch.cuda.empty_cache()
print('PREFLIGHT PASS — Cell T is armed')

In [ ]:
# C2b — WHERE DOES THE TIME GO. Run after a restart, before committing to Cell T.
# Splits one training step into: data / forward / backward / optimizer+governor.
import time, torch
from google.colab import userdata
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.model.governor import govern_model
from geolip.alephllm.train.optim import build_optimizers
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.data.tokenizers import build_tokenizer
import geolip.alephllm as A
print('package', A.__version__); assert tuple(int(x) for x in A.__version__.split('.')) >= (0,7,1), 'STALE RUNTIME'
p = get_preset('mini-beatrix-2'); cfg, tc = p.model, p.train
torch.manual_seed(0)
m = AlephLM(cfg).cuda(); m.train()
opts = build_optimizers(m, tc.muon_lr, tc.muon_momentum, tc.adam_lr)
def sync(): torch.cuda.synchronize()
x = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
# warmup (autotune, lazy inits)
with torch.autocast('cuda', dtype=torch.bfloat16):
    m(x, targets=x).loss.backward()
m.zero_grad(set_to_none=True); sync()
# (a) GPU compute, synthetic data: one full accumulation step
t0 = time.time(); tf = tb = 0.0
for _ in range(tc.grad_accum):
    s = time.time()
    with torch.autocast('cuda', dtype=torch.bfloat16):
        out = m(x, targets=x)
    sync(); tf += time.time() - s; s = time.time()
    (out.loss / tc.grad_accum).backward(); sync(); tb += time.time() - s
s = time.time()
torch.nn.utils.clip_grad_norm_(m.parameters(), tc.grad_clip)
for o in opts: o.step()
sync(); topt = time.time() - s
s = time.time(); hits = govern_model(m, tc.governor_theta); sync(); tgov = time.time() - s
m.zero_grad(set_to_none=True)
gpu_step = time.time() - t0
print(f'GPU step (synthetic data): {gpu_step:.1f}s = fwd {tf:.1f} + bwd {tb:.1f} + opt {topt:.1f} + governor {tgov:.2f} (hits {hits})')
print(f'  -> pure-compute ceiling: {tc.micro_batch*cfg.context*tc.grad_accum/gpu_step/1e3:.0f}k tok/s')
# (b) the data pipeline alone: how fast can the stream feed 2M tokens?
tok = build_tokenizer(cfg.tokenizer)
st = build_stream('wikitext-103', tok, cfg.context, tc.micro_batch, seed=1, role='train')
st.next_batch()  # open/download outside the clock
n_tok = 0; s = time.time()
while n_tok < 2_000_000:
    b = st.next_batch(); n_tok += b.numel()
dt = time.time() - s
print(f'stream: {n_tok/dt/1e3:.0f}k tok/s CPU -> {524288/(n_tok/dt):.1f}s of data per step if serial')
print('VERDICT: whichever line dominates the observed s/step is the bottleneck.')


In [ ]:
# C2c — COMPILE GATE (the utilization lever). The old Blackwell NaN verdict was
# measured on the v1 fused path; THIS batched graph has never been tested. Law:
# compile trains only if grad parity passes on the training hardware — the
# grad-vector comparison, not loss-only (loss-only gating shipped a NaN once).
import time, torch
from geolip.alephllm import get_preset
from geolip.alephllm.model.alephlm import AlephLM
p = get_preset('mini-beatrix-2'); cfg, tc = p.model, p.train
torch.manual_seed(0)
m = AlephLM(cfg).cuda(); m.train()
x = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
def grads(model):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        model(x, targets=x).loss.backward()
    g = {n: p.grad.detach().clone() for n, p in model.named_parameters() if p.grad is not None}
    model.zero_grad(set_to_none=True)
    return g
g_eager = grads(m)
mc = torch.compile(m)
with torch.autocast('cuda', dtype=torch.bfloat16):
    mc(x, targets=x).loss.backward()   # compile warmup (slow once)
m.zero_grad(set_to_none=True)
g_comp = {}
with torch.autocast('cuda', dtype=torch.bfloat16):
    mc(x, targets=x).loss.backward()
for n, prm in m.named_parameters():
    if prm.grad is not None: g_comp[n] = prm.grad.detach().clone()
m.zero_grad(set_to_none=True)
bad = [(n, float((g_eager[n] - g_comp[n]).abs().max())) for n in g_eager
       if not torch.isfinite(g_comp[n]).all() or
          float((g_eager[n] - g_comp[n]).abs().max()) > 1e-2 * (1e-6 + float(g_eager[n].abs().max()))]
nonfinite = [n for n in g_comp if not torch.isfinite(g_comp[n]).all()]
print('non-finite compiled grads:', nonfinite[:5] if nonfinite else 'NONE')
print('parity failures (rel>1e-2):', bad[:5] if bad else 'NONE')
torch.cuda.synchronize(); t0 = time.time()
for _ in range(3):
    with torch.autocast('cuda', dtype=torch.bfloat16):
        mc(x, targets=x).loss.backward()
    m.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sps = (time.time() - t0) / 3
print(f'compiled micro fwd+bwd: {sps:.2f}s -> projected step ~{sps*tc.grad_accum:.0f}s '
      f'({tc.micro_batch*cfg.context*tc.grad_accum/(sps*tc.grad_accum)/1e3:.0f}k tok/s)')
if not nonfinite and not bad:
    print('GATE PASS — set TrainConfig.compile=True for the session (p.train.compile = True before prepare)')
else:
    print('GATE FAIL — eager stands; report the lines above')


In [ ]:
# C2d — INSIDE ONE HUB: per-op forward timing on the real card. Names the slow op.
import time, torch, torch.nn.functional as F
from geolip.alephllm import get_preset
from geolip.alephllm.model.attention import CausalSplatHUB
from geolip.alephllm.model.bank import AnchoredBank
p = get_preset('mini-beatrix-2'); cfg, tc = p.model, p.train
torch.manual_seed(0)
hub = CausalSplatHUB(cfg.d_model, cfg.hub_K, cfg.hub_D, cfg.tau,
                     chunk=cfg.hub_chunk, n_const=cfg.hub_const).cuda()
bank = AnchoredBank(cfg.d_model, cfg.bank_experts, cfg.bank_ff, cfg.tau, cfg.gate_init).cuda()
B, n, d = tc.micro_batch, cfg.context, cfg.d_model
x = torch.randn(B, n, d, device='cuda')
def t(fn, reps=3):
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        fn(); torch.cuda.synchronize()
        s = time.time()
        for _ in range(reps): fn()
        torch.cuda.synchronize()
    return (time.time() - s) / reps
with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
    qc, kc = hub._code_cat_qk(x); v = hub.v(x)
    C = min(hub.chunk, n); pad = (-n) % C; nc = (n + pad) // C
    vc = (F.pad(v, (0,0,0,pad)) if pad else v).view(B, nc, C, d)
    mask = hub._mask(C, x.device, vc.dtype)
    K2 = qc.shape[-1]
    qv = (F.pad(qc, (0,0,0,pad)) if pad else qc).view(B, nc, C, K2)
    kv = (F.pad(kc, (0,0,0,pad)) if pad else kc).view(B, nc, C, K2)
    S = torch.einsum('bick,bicd->bikd', kv, vc)
    L = hub._prefix(nc, qc.device, qc.dtype)
    P = torch.matmul(L, S.reshape(B, nc, -1)).view_as(S)
    att = torch.einsum('bick,bijk->bicj', qv, kv) * mask
rows = [
 ('code_cat_qk (1 softmax)', t(lambda: hub._code_cat_qk(x))),
 ('v proj                 ', t(lambda: hub.v(x))),
 ('S einsum (write)       ', t(lambda: torch.einsum('bick,bicd->bikd', kv, vc))),
 ('prefix matmul P        ', t(lambda: torch.matmul(L, S.reshape(B, nc, -1)))),
 ('att within-chunk       ', t(lambda: torch.einsum('bick,bijk->bicj', qv, kv) * mask)),
 ('num read (qc.P + att.v)', t(lambda: torch.einsum('bick,bikd->bicd', qv, P) + att @ vc)),
 ('FULL hub.forward       ', t(lambda: hub(x))),
 ('bank.forward           ', t(lambda: bank(x))),
]
for name, dt in rows: print(f'{name} {dt*1e3:8.1f} ms')
d_ = dict(rows)
print(f"-> 32 layers x (hub+bank) = {(d_['FULL hub.forward       '] + d_['bank.forward           '])*32:.1f}s per micro-forward")
print(f'-> dtypes: qc {qc.dtype}, v {v.dtype}  (bf16 expected everywhere now)')

In [ ]:
# C2e — hub_chunk sweep on the real card: S/P traffic ~ 1/C, att FLOPs ~ C.
# 4090 evidence (2026-08-26): flat 256–1024 wall-clock, peak memory favors larger.
# The Blackwell rules; if a non-256 chunk wins >10%%, set p.model.hub_chunk before
# build (config field, no arch change, checkpoint-compatible).
import time, torch
from geolip.alephllm import get_preset
from geolip.alephllm.model.attention import CausalSplatHUB
p = get_preset('mini-beatrix-2'); cfg, tc = p.model, p.train
torch.manual_seed(0)
x = torch.randn(tc.micro_batch, cfg.context, cfg.d_model, device='cuda')
for C in (256, 512, 768, 1024, 1536):
    hub = CausalSplatHUB(cfg.d_model, cfg.hub_K, cfg.hub_D, cfg.tau,
                         chunk=C, n_const=cfg.hub_const).cuda()
    def run():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = hub(x).square().mean()
        loss.backward(); hub.zero_grad(set_to_none=True)
    run(); torch.cuda.synchronize()
    s = time.time()
    for _ in range(3): run()
    torch.cuda.synchronize()
    print(f'chunk {C:5d}: {(time.time()-s)/3*1e3:7.1f} ms fwd+bwd/layer  '
          f'peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB')
    torch.cuda.reset_peak_memory_stats()
    del hub; torch.cuda.empty_cache()

In [ ]:
# C3 (T) — THE SESSION. Resume-first; stops at stage boundaries; every boundary
# ships a checkpoint (trainer HubSync) AND a report JSON (this cell). Interrupt-safe.
import gc, io, json, time, torch
from google.colab import userdata
from geolip.alephllm import prepare
from geolip.alephllm.data import curriculum as C
from geolip.alephllm.train import probes
from geolip.alephllm.train.instruments import model_census, toggle_ledger, readout
from geolip.alephllm.model.governor import govern_model

PRESET = 'mini-beatrix-2'
MAX_HOURS = 10.5

run = prepare(PRESET, hf_token=userdata.get('HF_TOKEN'))
C.append_curriculum_phases(run.manifest)            # idempotent: adds S0–S8 after the pretrain phases

def boundary_report(run, tag):
    m, tok, dev = run.raw_model, run.tokenizer, run.device
    m.eval()
    with torch.no_grad():
        pr = probes.run_all(m, tok, dev)
        sample = torch.randint(0, 255, (2, min(1024, m.cfg.context)), device=dev)
        census = model_census(m, sample)
    rep = {'tag': tag, 'step': run.step, 'tokens': run.manifest.tokens_seen,
           'probes': pr, 'census_flags': census.get('flags'),
           'governor_hits_cum': getattr(run, '_gov_hits', 0),
           'crowd_check_extra_hits': govern_model(m, run.tc.governor_theta) if run.tc.governor else None}
    run.hub.upload_bytes(json.dumps(rep, default=float).encode(),
                         f'{run.preset.prefix}/reports/v2/{tag}_step{run.step}.json')
    print(probes.report(pr)); m.train()
    return rep

t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    hours_left = (t_end - time.time()) / 3600
    if hours_left < 0.2: break
    phase_before = run.manifest.current_phase()['name'] if run.manifest.current_phase() else None
    run.train(max_hours=hours_left)                 # returns at boundary, cap, or completion
    tag = phase_before or 'final'
    boundary_report(run, tag)
    if run.manifest.current_phase() is None:
        print('curriculum complete'); break
    gc.collect(); torch.cuda.empty_cache()
print('session over — resume state on the hub')

In [ ]:
# C4 — growth table from the hub's v2 boundary reports.
import json
from huggingface_hub import HfApi, hf_hub_download
from geolip.alephllm.presets import TRAINING_REPO
PRESET = 'mini-beatrix-2'
api = HfApi()
files = sorted(f for f in api.list_repo_files(TRAINING_REPO)
               if f.startswith(f'{PRESET}/reports/v2/') and f.endswith('.json'))
rows = []
for f in files:
    r = json.load(open(hf_hub_download(TRAINING_REPO, f)))
    rows.append((r['tag'], r['step'], round(r['tokens']/1e9, 2),
                 {k: round(v, 3) for k, v in r['probes'].items() if isinstance(v, float)},
                 r.get('governor_hits_cum')))
print(f"{'stage':<16}{'step':>8}{'tokens(B)':>10}  probes · governor hits")
for tag, step, tk, pr, gh in rows:
    print(f'{tag:<16}{step:>8}{tk:>10}  {pr} · gov {gh}')

### Notes
- **Screen first**: `mini-beatrix-2s` runs every gating cell at 3.6× less cost; the big craft inherits verdicts, not hopes.
- **Controls**: `mini-beatrix-2-control` (pure sdpa) and `governor=''` (set on a copied preset) are the two arms any v2 claim must beat/tie under Law E bars (2 seeds; the seed field lives in `TrainConfig.seed`).
- **Monitoring keys**: `governor/hits_cum` (TB) — expect 0 early, small and rising only if crowding pressure exists; `hub_addr.anchor_merge_pairs` must stay 0 (the governor's whole job); `hub_consumed_erank` per block is the supply-utilization gauge (v1's L4 read 2 — the disease this design retires).
- **Kernels**: eager-fused path only on Blackwell (compiled bf16 backward NaNs — measured); `hub_chunk=256`; fp16 forbidden.
- Sessions are boundary-exact: interrupting mid-stage is safe (resume-first), but reports only ship at boundaries.